# Harmonized DC Hydrology for All Stations

This notebook applies **one standard method** to every station, replacing the station-by-station pilot code in
`01_dc_hydrology_pilot.ipynb`. To add stations (for the national run), only the `STATIONS` dictionary changes.

## Standard method

| Step | Rule |
|---|---|
| Basin | USGS NLDI upstream basin polygon (`/nwissite/USGS-<station>/basin`) |
| Basin check | Basin area ÷ official USGS drainage area should be ≈ 1.00 |
| HUC12 layer | October-2020 WBD (`102020_wbd_hu12.gpkg`), the version the DC is mapped to |
| HUC12 inclusion | Keep HUC12s with **≥ 1 km² inside** the basin (drops edge slivers) |
| DC product | `wqn-nhmprms-conus-nwaa-v1`, variables `incqkflow`, `incbsflow`, `soilmstfr` |
| Window | 2013-01 to 2021-08 (104 months) |
| Aggregation | **Area-weighted** mean: weight = HUC12 area inside ÷ total inside area |
| Output columns | `station_id` (8-digit text), `year_month` (`YYYY-MM`), `dc_incqkflow_mm`, `dc_incbsflow_mm`, `dc_soilmstfr` |
| Join | `station_id` + `year_month` of the sampling date; left join; every pesticide row kept |

## Sections
1. Setup and configuration
2. Basins (NLDI) and basin check
3. HUC12 weights
4. DC extraction (one pass over the 5.3 GB file for all stations)
5. Area-weighted basin-month values
6. Join to pesticide observations
7. QC summary

## 1. Setup and configuration

Mount Google Drive, load the USGS API key from Colab **Secrets** (name: `USGS_API_KEY`; never type the key in a
cell), and set paths. All outputs are written to `OUT_DIR` in Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
try:
    from google.colab import userdata
    os.environ["USGS_API_KEY"] = userdata.get("USGS_API_KEY") or ""
except Exception:
    os.environ.setdefault("USGS_API_KEY", "")
print("API key loaded:", bool(os.environ["USGS_API_KEY"]))

In [ ]:
import os
import time
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd

# ---------------- stations (add national sites here) ----------------
STATIONS = {
    "06800000": "Maple Creek",
    "05451210": "South Fork Iowa River",
    "02091500": "Contentnea Creek",
}

# ---------------- paths ----------------
DRIVE = Path("/content/drive/MyDrive/Datasets")
WBD_FILE = DRIVE / "102020_wbd_hu12.gpkg"                   # Oct-2020 HUC12 layer
PESTICIDE_FILE = DRIVE / "t2_pest_concs_enriched_v2.parquet"
OUT_DIR = DRIVE / "dc_harmonized"
BASIN_DIR = OUT_DIR / "basins"
for d in (OUT_DIR, BASIN_DIR):
    d.mkdir(parents=True, exist_ok=True)

# DC combined file: URL, or a local copy (much faster), e.g. "/content/combined_...long.csv"
DC_SOURCE = ("https://water.usgs.gov/nwaa-data/data/water-quantity/wqn-nhmprms-conus-nwaa-v1/"
             "combined_wqn-nhmprms-conus-nwaa-v1_CONUS_198301-202109_long.csv")

# ---------------- method settings ----------------
VARS = {                     # DC variable -> standard output column
    "incqkflow": "dc_incqkflow_mm",
    "incbsflow": "dc_incbsflow_mm",
    "soilmstfr": "dc_soilmstfr",
}
START, END = "2013-01", "2021-08"
N_MONTHS = len(pd.period_range(START, END, freq="M"))     # 104
MIN_INSIDE_KM2 = 1.0          # HUC12 inclusion rule
AREA_RATIO_TOL = 0.05         # flag basins whose area differs >5% from the official drainage area
EA = "EPSG:5070"              # equal-area projection for area calculations

NLDI = "https://api.water.usgs.gov/nldi/linked-data"
SITE_API = "https://api.waterdata.usgs.gov/ogcapi/v0/collections/monitoring-locations/items"
HEADERS = {"User-Agent": "nwaa-dc-pesticide/1.0"}
if os.environ.get("USGS_API_KEY"):
    HEADERS["X-Api-Key"] = os.environ["USGS_API_KEY"]

print(f"{len(STATIONS)} stations | window {START} to {END} ({N_MONTHS} months)")

## 2. Basins (NLDI) and basin check

For each station, download the NLDI upstream basin (or reuse a saved copy) and compare its area with the
official USGS drainage area. A ratio near 1.00 means the station snapped to the correct river. Requests are
retried when the USGS server is busy (502/503/504).

In [ ]:
def get_json(url, params=None, tries=4):
    """GET with retries for temporary USGS server errors."""
    for i in range(tries):
        r = requests.get(url, params=params, headers=HEADERS, timeout=180)
        if r.status_code not in (502, 503, 504):
            r.raise_for_status()
            return r.json()
        print(f"  server busy ({r.status_code}), retry {i + 1}/{tries}")
        time.sleep(5 * (i + 1))
    r.raise_for_status()


def get_basin(station):
    """NLDI upstream basin polygon (cached in BASIN_DIR)."""
    path = BASIN_DIR / f"{station}_basin.geojson"
    if not path.exists():
        j = get_json(f"{NLDI}/nwissite/USGS-{station}/basin", {"f": "json"})
        gpd.GeoDataFrame.from_features(j["features"], crs="EPSG:4326").to_file(path, driver="GeoJSON")
    return gpd.read_file(path).dissolve()[["geometry"]]


def official_drainage_km2(station):
    """Official USGS drainage area (mi² -> km²); None if unavailable."""
    try:
        j = get_json(f"{SITE_API}/USGS-{station}", {"f": "json"})
        da = j["properties"].get("drainage_area")
        return float(da) * 2.58999 if da else None
    except Exception as e:
        print(f"  drainage area lookup failed for {station}: {e}")
        return None


basins, basin_qc = {}, []
for st, name in STATIONS.items():
    b = get_basin(st)
    basins[st] = b
    km2 = b.to_crs(EA).area.iloc[0] / 1e6
    da = official_drainage_km2(st)
    ratio = km2 / da if da else None
    basin_qc.append({
        "station_id": st, "station_name": name,
        "basin_km2": round(km2, 1),
        "official_km2": round(da, 1) if da else None,
        "area_ratio": round(ratio, 3) if ratio else None,
        "area_flag": ratio is None or abs(ratio - 1) > AREA_RATIO_TOL,
    })

basin_qc = pd.DataFrame(basin_qc)
basin_qc

## 3. HUC12 weights

Overlay each basin on the Oct-2020 HUC12 layer (in an equal-area projection), measure the area of each HUC12
inside the basin, keep HUC12s with ≥ 1 km² inside, and compute weights that sum to 1.
HUC12 IDs are stored as 12-character text so leading zeros (e.g., North Carolina `03…`) are kept.

In [ ]:
def huc12_weights(station, basin):
    """HUC12s inside the basin with area weights (standard inclusion rule)."""
    huc = gpd.read_file(WBD_FILE, bbox=basin)          # only HUC12s near this basin
    col = next(c for c in huc.columns if c.lower() in ("huc12", "huc12_id"))
    huc = huc.rename(columns={col: "huc12"})
    huc["huc12"] = huc["huc12"].astype(str).str.strip().str.zfill(12)

    b, h = basin.to_crs(EA), huc.to_crs(EA)
    h["huc_km2"] = h.area / 1e6
    ov = gpd.overlay(b, h[["huc12", "huc_km2", "geometry"]], how="intersection")
    ov["inside_km2"] = ov.area / 1e6
    t = ov.groupby(["huc12", "huc_km2"], as_index=False)["inside_km2"].sum()
    t["pct_of_huc_inside"] = 100 * t.inside_km2 / t.huc_km2

    n_touching = len(t)
    t = t[t.inside_km2 >= MIN_INSIDE_KM2].copy()
    t["weight"] = t.inside_km2 / t.inside_km2.sum()
    t.insert(0, "station_id", station)
    t.attrs["n_touching"] = n_touching
    return t.sort_values("weight", ascending=False).reset_index(drop=True)


weights, huc_qc = [], []
for st, b in basins.items():
    t = huc12_weights(st, b)
    basin_km2 = basin_qc.set_index("station_id").loc[st, "basin_km2"]
    huc_qc.append({
        "station_id": st,
        "huc12_touching": t.attrs["n_touching"],
        "huc12_kept": len(t),
        "coverage_pct": round(100 * t.inside_km2.sum() / basin_km2, 1),
        "weight_sum": round(t.weight.sum(), 6),
    })
    weights.append(t)
    print(f"\n{st} {STATIONS[st]}: {len(t)} HUC12s kept of {t.attrs['n_touching']} touching")
    print(t.round(3).to_string(index=False))

weights = pd.concat(weights, ignore_index=True)
weights.to_csv(OUT_DIR / "huc12_weights_all_stations.csv", index=False)
huc_qc = pd.DataFrame(huc_qc)
huc_qc

## 4. DC extraction (single pass)

The combined NHM-PRMS file holds all CONUS HUC12s × 465 months × 8 variables. It is read **once**, in chunks,
keeping only the union of HUC12s from all stations, the analysis window, and the three variables. Columns are
selected by name, not position (this file lists `year_month` before `huc12_id`).

This takes several minutes over the internet; copying the file to `/content/` first and setting `DC_SOURCE` to
that path is much faster.

In [ ]:
def extract_dc(source, huc12_ids, start=START, end=END, chunksize=2_000_000):
    """Stream the combined DC file and keep selected HUC12s, months and variables."""
    header = pd.read_csv(source, nrows=0).columns.tolist()
    hcol = next(c for c in header if c.lower().startswith("huc12"))
    ycol = next(c for c in header if c.lower() == "year_month")
    vcols = {next(c for c in header if c.split("_")[0] == v): out for v, out in VARS.items()}
    print("Using columns:", [hcol, ycol] + list(vcols))

    s, e = start.replace("-", ""), end.replace("-", "")
    kept, n = [], 0
    for ch in pd.read_csv(source, usecols=[hcol, ycol] + list(vcols),
                          dtype={hcol: str, ycol: str}, chunksize=chunksize):
        n += len(ch)
        ch[hcol] = ch[hcol].str.strip().str.zfill(12)
        ym = ch[ycol].str.replace(r"[^0-9]", "", regex=True).str[:6]
        mask = ch[hcol].isin(huc12_ids) & ym.between(s, e)
        if mask.any():
            part = ch.loc[mask, [hcol] + list(vcols)].copy()
            part.insert(1, "year_month", ym[mask].str[:4] + "-" + ym[mask].str[4:])
            kept.append(part)
        print(f"  scanned {n:,} rows", end="\r")

    out = (pd.concat(kept, ignore_index=True)
             .rename(columns={hcol: "huc12", **vcols})
             .sort_values(["huc12", "year_month"])
             .reset_index(drop=True))
    print(f"\nKept {len(out):,} rows")
    return out


ALL_HUC12 = set(weights.huc12)
huc12_month = extract_dc(DC_SOURCE, ALL_HUC12)
huc12_month.to_csv(OUT_DIR / "dc_huc12_month_all_stations.csv", index=False)

missing_huc = ALL_HUC12 - set(huc12_month.huc12)
print("HUC12s requested:", len(ALL_HUC12), "| found:", huc12_month.huc12.nunique(),
      "| missing:", missing_huc or "none")
print("Rows:", len(huc12_month), "| expected:", len(ALL_HUC12) * N_MONTHS)
print("Missing values:", int(huc12_month[list(VARS.values())].isna().sum().sum()))

## 5. Area-weighted basin-month values

For station $s$ and month $t$:

$$\text{basin value}_{s,t} = \frac{\sum_i w_{s,i}\,x_{i,t}}{\sum_i w_{s,i}}$$

where $w_{s,i}$ is the weight of HUC12 $i$ in basin $s$, summed over HUC12s that have a value that month.
`dc_coverage` reports the share of the basin weight with data (should be 1.0).

In [ ]:
def basin_month_values(weights, huc12_month):
    """Area-weighted mean of each DC variable per station and month."""
    dc_cols = list(VARS.values())
    m = weights[["station_id", "huc12", "weight"]].merge(huc12_month, on="huc12", how="inner")
    rows = []
    for (st, ym), g in m.groupby(["station_id", "year_month"]):
        row = {"station_id": st, "year_month": ym}
        for c in dc_cols:
            ok = g[c].notna()
            row[c] = (g.weight[ok] * g[c][ok]).sum() / g.weight[ok].sum() if ok.any() else pd.NA
        row["dc_coverage"] = round(g.weight[g[dc_cols].notna().all(axis=1)].sum(), 6)
        row["dc_n_huc12"] = g.huc12.nunique()
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(["station_id", "year_month"]).reset_index(drop=True)
    out.insert(1, "station_name", out.station_id.map(STATIONS))
    return out


basin_month = basin_month_values(weights, huc12_month)
basin_month.to_csv(OUT_DIR / "dc_basin_month_all_stations.csv", index=False)

check = basin_month.groupby("station_id").agg(
    months=("year_month", "nunique"),
    min_coverage=("dc_coverage", "min"),
    missing_values=("dc_incqkflow_mm", lambda x: int(x.isna().sum())),
)
check["expected_months"] = N_MONTHS
print(check)
assert (check.months == N_MONTHS).all(), "Some stations do not have all months"
basin_month.head()

## 6. Join to pesticide observations

- `station_id` = 8-digit USGS number at the end of `Pesticide_Site` (e.g., `Atrazine_06800000` → `06800000`).
- `year_month` = month of `Activity_StartDate`; exact sampling dates and times are unchanged.
- Only the selected stations and the analysis window are kept; all analytes are retained.
- Left join, many samples → one basin-month (validated). Checks: row count unchanged, no missing DC values.
- DC columns keep the `dc_` prefix, so modeled hydrology stays distinct from the observed variables.

In [ ]:
pest = pd.read_parquet(PESTICIDE_FILE)
print("Pesticide dataset:", pest.shape)

pest["station_id"] = (pest["Pesticide_Site"].astype(str)
                      .str.extract(r"_(\d+)$")[0].str.zfill(8))
pest["year_month"] = pd.to_datetime(pest["Activity_StartDate"], errors="coerce").dt.strftime("%Y-%m")

sel = pest[pest.station_id.isin(STATIONS) & pest.year_month.between(START, END)].copy()
print("Rows for selected stations in window:", len(sel))
print("Rows with unreadable dates:", int(pest.year_month.isna().sum()))

dc_join = basin_month[["station_id", "year_month"] + list(VARS.values())]
combined = sel.merge(dc_join, on=["station_id", "year_month"], how="left",
                     validate="many_to_one", indicator=True)

assert len(combined) == len(sel), "Join changed the number of rows"
unmatched = combined[combined["_merge"] != "both"]
print("Rows without DC match:", len(unmatched))
combined = combined.drop(columns="_merge")

combined.to_parquet(OUT_DIR / "pesticide_with_dc_all_stations.parquet", index=False)
for st in STATIONS:
    combined[combined.station_id == st].to_csv(OUT_DIR / f"combined_{st}_with_dc.csv", index=False)
print("Saved combined tables to", OUT_DIR)

## 7. QC summary

One table per station for the professor's summary: basin check, HUC12 coverage, DC completeness, and join results.

In [ ]:
join_qc = combined.groupby("station_id").agg(
    pesticide_rows=("year_month", "size"),
    analytes=("Pesticide_Name", "nunique"),
    sampling_months=("year_month", "nunique"),
    rows_missing_dc=("dc_incqkflow_mm", lambda x: int(x.isna().sum())),
).reset_index()

qc = (basin_qc.merge(huc_qc, on="station_id")
              .merge(check.reset_index()[["station_id", "months", "min_coverage"]], on="station_id")
              .merge(join_qc, on="station_id", how="left"))
qc.to_csv(OUT_DIR / "qc_summary.csv", index=False)
qc

### Spot check

Samples in the same station-month must carry identical DC values, and a few joined rows are recomputed directly
from the HUC12 table.

In [ ]:
# 1. One set of DC values per station-month
per_month = combined.groupby(["station_id", "year_month"])[list(VARS.values())].nunique()
print("Station-months with more than one DC value:", int((per_month > 1).any(axis=1).sum()))

# 2. Recompute a few joined values independently
sample = combined.dropna(subset=["dc_incqkflow_mm"]).groupby("station_id").sample(3, random_state=1)
for _, r in sample.iterrows():
    w = weights[weights.station_id == r.station_id]
    x = huc12_month[huc12_month.year_month == r.year_month].merge(w, on="huc12")
    recomputed = (x.weight * x.dc_incqkflow_mm).sum() / x.weight.sum()
    print(f"{r.station_id} {r.year_month}: joined {r.dc_incqkflow_mm:.4f} | recomputed {recomputed:.4f}")